# Modeling, Validation, and Market Analysis

The goal is not only to produce a price estimate, but to support a practical decision: how does an advertised price compare with the market evidence available for a vehicle? This notebook trains a model, checks it against a simple baseline, examines its errors, and prepares examples that can be reviewed by a hiring team.



In [14]:
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)


## 1. Load the cleaned data and define the features

In [15]:
df = pd.read_csv("cars_cleaned.csv")

features = [
    "brand", "model", "trim", "year",
    "mileage_km", "city", "body_condition"
]
target = "price_toman"

X = df[features].copy()
y = df[target].copy()

print("Dataset shape:", df.shape)
print("Features:", features)
print("Target:", target)


Dataset shape: (3867, 9)
Features: ['brand', 'model', 'trim', 'year', 'mileage_km', 'city', 'body_condition']
Target: price_toman


## 2. Create a train/test split

### do it on 80/20

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


## 3. Preprocessing and Random Forest model

The preprocessing steps and estimator are combined in a single pipeline. This ensures that imputers and encoders are fitted on the training data only and then applied consistently to the test data and future API requests. Fitting preprocessing on the full dataset before splitting would leak information from the test set.

for this dataset, Random Forest works because its a reasonable baseline for mixed numeric and categorical data.

In [17]:
categorical_features = [
    "brand", "model", "trim", "city", "body_condition"
]
numeric_features = ["year", "mileage_km"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_features,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=150,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )),
])


## 4. Evaluate the model with held-out metrics

in this project, i'd rather to use R2, MAE and RMSE

In [18]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.2f} million Toman")
print(f"RMSE: {rmse:.2f} million Toman")
print(f"R²:   {r2:.3f}")


MAE:  786.86 million Toman
RMSE: 2250.54 million Toman
R²:   0.745


seems it would be less accurate for super cars or very cheap cars!
but it suppose to work well for everyday cars

## 5. Compare against a simple baseline

A model score is more meaningful when compared with a simple alternative. The baseline predicts a central training-set price for every test row, giving a reference point that does not use vehicle details.

If the Random Forest does not improve on this baseline, the extra complexity is not justified. If it does improve, that is evidence that the selected vehicle features contain useful signal.

In [19]:
train_medians = (
    X_train.assign(price=y_train)
    .groupby(["brand", "model"])["price"]
    .median()
)

global_median = y_train.median()

baseline_predictions = pd.Series(
    [
        train_medians.get((brand, model), global_median)
        for brand, model in zip(X_test["brand"], X_test["model"])
    ],
    index=y_test.index,
)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Baseline MAE:      {baseline_mae:.2f}")
print(f"Random Forest MAE: {mae:.2f}")
print(f"MAE improvement:   {(1 - mae / baseline_mae) * 100:.1f}%")


Baseline MAE:      982.39
Random Forest MAE: 786.86
MAE improvement:   19.9%


# as the result, RF is notably better. so its reasonable to use that.

## 6. Inspect the largest prediction errors


In [21]:
comparison = X_test.copy()
comparison["actual_price"] = y_test
comparison["predicted_price"] = predictions
comparison["absolute_error"] = (comparison["actual_price"] - comparison["predicted_price"]).abs()
comparison["relative_error_pct"] = (
    comparison["absolute_error"] / comparison["actual_price"].replace(0, np.nan) * 100
)

print("Largest absolute errors:")
display(comparison.sort_values("absolute_error", ascending=False).head(15))



Largest absolute errors:


,brand,model,trim,year,mileage_km,city,body_condition,actual_price,predicted_price,absolute_error,relative_error_pct
1128,تویوتا,پرادو چهار در,2.4 لیتر توربو,1405,0,تهران,بدون رنگ,48000.0,24163.070394,23836.929606,49.660270
3137,ب ام و,X3,28,1396,11000,تهران,تعویض گلگیر,40000.0,19438.746825,20561.253175,51.403133
1068,تویوتا,پرادو چهار در,2.4 لیتر توربو,1405,0,تهران,بدون رنگ,44000.0,24163.070394,19836.929606,45.083931
1221,تویوتا,کرولا,1.2 لیتر توربو,1404,0,تهران,بدون رنگ,7850.0,26018.481767,18168.481767,231.445628
937,تویوتا,لوین هیبرید,1.8 لیتر ساده,1404,0,تهران,بدون رنگ,8400.0,26032.575274,17632.575274,209.911610
1226,فیات,دوکاتو,دنده ای,1397,0,تهران,بدون رنگ,24000.0,9004.920691,14995.079309,62.479497
1052,تویوتا,پرادو چهار در,2.4 لیتر توربو,1404,0,تهران,بدون رنگ,40500.0,26163.583607,14336.416393,35.398559
1228,تویوتا,وایلدلندر,2.0 لیتر AWD لاکچری پلاس,1404,0,تهران,بدون رنگ,13300.0,26032.575274,12732.575274,95.733649
3049,بنز,کلاس S,S500 هشت سیلندر,1391,185000,تهران,تعویض کاپوت,27000.0,14556.647302,12443.352698,46.086491
1159,کیا,اسپورتیج,4 سیلندر توربو,1404,0,تهران,بدون رنگ,890.0,12553.590419,11663.590419,1310.515777


### as it was predicted, the most errors happend in super cars which we had fewer cases to train the model

## 7. Select comparable vehicles

collect 5 cars as descision supporters based on Brand & Model

In [9]:
def find_comparables(car, data, n=5):
    work = data.copy()

    # Prefer same brand/model, then same trim if available.
    same_model = work[
        (work["brand"] == car["brand"]) &
        (work["model"] == car["model"])
    ].copy()

    pool = same_model if len(same_model) >= 3 else work.copy()

    if len(pool) == 0:
        return pool

    score = np.zeros(len(pool), dtype=float)

    score += np.where(pool["brand"].eq(car["brand"]), 5, 0)
    score += np.where(pool["model"].eq(car["model"]), 10, 0)
    score += np.where(pool["trim"].eq(car["trim"]), 4, 0)
    score += np.where(pool["city"].eq(car["city"]), 2, 0)
    score += np.where(pool["body_condition"].eq(car["body_condition"]), 2, 0)

    score -= (pool["year"].astype(float) - float(car["year"])).abs() * 0.8
    score -= (pool["mileage_km"].astype(float) - float(car["mileage_km"])).abs() / 50000.0

    pool = pool.assign(comparable_score=score)
    cols = [
        "brand","model","trim","year","mileage_km",
        "city","body_condition","price_toman","comparable_score"
    ]
    return pool.sort_values("comparable_score", ascending=False).head(n)[cols]

# Example using the first test row.
example_car = X_test.iloc[0].to_dict()
display(find_comparables(example_car, df, n=5))


,brand,model,trim,year,mileage_km,city,body_condition,price_toman,comparable_score
2580,رنو,تندر 90,اتوماتیک,1396,137000,تهران,ترمیم,1980.0,23.00
2485,رنو,تندر 90,اتوماتیک,1396,153000,تهران,ترمیم,2200.0,22.68
1467,رنو,تندر 90,اتوماتیک,1396,123000,تهران,چند لکه رنگ,2430.0,20.72
1697,رنو,تندر 90,اتوماتیک,1396,162000,تهران,چند لکه رنگ,2150.0,20.50
605,رنو,تندر 90,اتوماتیک,1394,190000,تهران,یک لکه رنگ,1950.0,18.34


## 8. Explain the model with aggregate feature importance


In [10]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
importances = rf.feature_importances_

importance_df = pd.DataFrame({
    "encoded_feature": feature_names,
    "importance": importances
})

def original_feature(name):
    if name.startswith("categorical__"):
        encoded = name.replace("categorical__", "")
        return encoded.split("_", 1)[0]
    if name.startswith("numeric__"):
        return name.replace("numeric__", "")
    return name

importance_df["feature_group"] = importance_df["encoded_feature"].map(original_feature)

grouped_importance = (
    importance_df.groupby("feature_group")["importance"]
    .sum()
    .sort_values(ascending=False)
    .to_frame()
)

display(grouped_importance)


,importance
feature_group,
brand,0.328903
model,0.204453
trim,0.165870
mileage_km,0.157348
year,0.100885
body,0.022754
city,0.019788


#### Aggregate feature importance describes model behavior; it does not establish causality.

## 9. Five examples
### these will be saved into a csv

In [12]:
def model_output(test_index):
    car = X_test.loc[test_index].to_dict()
    row = pd.DataFrame([car])
    estimated = float(model.predict(row)[0])
    transformed = model.named_steps["preprocessor"].transform(row)
    tree_values = np.array([tree.predict(transformed)[0] for tree in rf.estimators_])
    low, high = np.percentile(tree_values, [10, 90])
    asking = float(y_test.loc[test_index])
    diff = (asking - estimated) / estimated * 100 if estimated > 0 else np.nan
    position = "Below Market" if diff < -5 else ("Above Market" if diff > 5 else "Fair")
    return {
        **car,
        "asking_price": round(asking, 2),
        "estimated_price": round(estimated, 2),
        "price_range_p10_p90": [round(float(low), 2), round(float(high), 2)],
        "difference_percent": round(float(diff), 2),
        "market_position": position,
    }

five_outputs = [model_output(idx) for idx in X_test.index[:5]]
five_outputs_df = pd.DataFrame(five_outputs)
display(five_outputs_df)
five_outputs_df.to_csv("five_real_test_outputs.csv", index=False, encoding="utf-8-sig")
print("Saved: five_real_test_outputs.csv")


,brand,model,trim,year,mileage_km,city,body_condition,asking_price,estimated_price,price_range_p10_p90,difference_percent,market_position
0,رنو,تندر 90,اتوماتیک,1396,137000,تهران,ترمیم,1980.0,2493.57,"[1878.33, 3225.0]",-20.60,Below Market
1,پژو,206,تیپ 2,1385,520000,تهران,تمام رنگ,620.0,651.01,"[524.5, 803.67]",-4.76,Fair
2,تویوتا,bZ3,الیت پرو,1403,11000,تهران,یک لکه رنگ,6990.0,13163.29,"[5550.0, 16983.75]",-46.90,Below Market
3,سیتروئن,زانتیا,2.0 لیتر SX,1382,300000,زنجان,دوررنگ,500.0,743.22,"[550.0, 1029.5]",-32.73,Below Market
4,نیسان,سیلفی e-POWER,پلاس,1403,0,تهران,بدون رنگ,8700.0,9494.82,"[7199.0, 13403.33]",-8.37,Below Market


Saved: five_real_test_outputs.csv


## 13. Save the trained model and supporting artifacts

Save the fitted pipeline and any metadata required by the API.

In [23]:
joblib.dump(model, "car_price_model.joblib")
print("Saved: car_price_model.joblib")


Saved: car_price_model.joblib


## 14. Interpretation and limitations

- **Asking Price** is the seller's advertised amount. **Transaction Price** is the amount actually paid after negotiation; it is not observed in this dataset.
- **Fair Market Value** is an approximate listing-based estimate here, not a verified transaction value.
- The reported range comes from Random Forest tree dispersion. It is a practical uncertainty signal, not a calibrated statistical prediction interval.
- Comparable listings support the decision but do not prove the price a buyer and seller will agree on.
- Rare models, unusual trims, super cars, and categories with few examples can have substantially larger errors.
- Useful next steps include rolling or periodic retraining, calibrated prediction intervals, improved comparable retrieval, and access to verified transaction prices.